# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.


# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `


In [2]:
import json
import tiktoken
#import pandas as pd||
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os

C:\Users\m\AppData\Local\Temp\ipykernel_10736\4814559.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader


##  LLM

In [3]:
load_dotenv(override=True)

True

In [4]:
import os
from langchain_ollama import ChatOllama

load_dotenv(override=True)

api_key = os.getenv("OLLAMA_API_KEY")

llm = ChatOllama(
    model="gemma4:31b",   
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": f"Bearer {api_key}"}},
    temperature=0,
)

# 1. Loading the pdf, Chunking

Je vais créer un dossier que je vais appeler `PDFs` et à l'intérieur duquel je vais coller ce fichier PDF `nc346.pdf` (pour vous, choisissez votre CV par exemple, ou un fichier que vous connaissez déjà).


In [41]:
pdf_file="./pdfs/Chapitre_3_Prétraitement_des_documents_textes.pdf"

In [42]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On va utiliser le découpage (*split*). Je rappelle donc que l'on utilise `RecursiveCharacterTextSplitter` avec l'encodeur `tiktoken`. Voici le modèle Tiktoken utilisé : la taille de chaque bloc (*chunk*) est de 300 tokens et le chevauchement (*overlap*) est de 20.

Maintenant, je vais charger le modèle `TextSplitter`. Notez qu'ici, il manque une dépendance. Pour effectuer le découpage, le système vous demande d'installer `pypdf`.

Faites donc : `uv add pypdf`.


In [43]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

Dans le terminal de projet ajouter ` uv add pypdf `

La méthode `load_and_split` va charger le fichier, puis appliquer cette stratégie de découpage. Chaque bloc (*chunk*) aura une taille de 300 tokens avec un chevauchement (*overlap*) de 20. Si vous souhaitez vérifier le nombre total de blocs générés, vous pouvez utiliser `len(chunks)`.


In [44]:

chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)


36

# 2.vectore Store - ChromaDB et Embedding

 *Importer le model d'embedding* 



In [9]:
# from langchain_openai import OpenAIEmbeddings
# embedding_model=OpenAIEmbeddings(model='text-embedding-3-small')
#text-embedding-3-small
#text-embedding-ada-002

# import os
# from dotenv import load_dotenv
# from langchain_ollama import OllamaEmbeddings

# load_dotenv(override=True)
# api_key = os.getenv("OLLAMA_API_KEY")

# embedding_model = OllamaEmbeddings(
#     model="embeddinggemma",
#     base_url="https://ollama.com",
#     client_kwargs={"headers": {"Authorization": f"Bearer {api_key}"}},
# )

# print(len(embedding_model.embed_query("test")))


from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
print(len(embedding_model.embed_query("test")))  # 1024

c:\AcadimicProjects\python_uv_workspace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4082.02it/s]


1024


Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

In [45]:
vectorstore=Chroma.from_documents(chunks,embedding_model,collection_name="nc346_v3",persist_directory="./store")

# 3.  **Trouver les chunks liée à la questions (Retriever)**

In [46]:
#crée le Retriever 
retriever=vectorstore.as_retriever(search_type='similarity',search_Kwargs={'k',5})
# k=5. Donc je vais lui demander de me chercher les cinq chunks les plus proches à la question.


On va faire un test. J'ai la question suivante : 
> « Quelle est la capitalisation boursière du Maroc pour 2025 ? »

La méthode `retriever.invoke()` a recherché les blocs (*chunks*). Je vais maintenant les afficher.


In [47]:
#test
retrieved_chunks=retriever.invoke(" Qui sont les techniques de prétraitement de text?")
print(retrieved_chunks)
print("nombre de chunks est :",len(retrieved_chunks))


[Document(metadata={'author': 'INFORMATIQUE', 'company': 'Microsoft', 'moddate': '2023-10-30T09:46:34+01:00', 'page_label': '5', 'title': 'Chapitre1 Introduction aux systèmes de reconnaissances de des formes', 'source': './pdfs/Chapitre_3_Prétraitement_des_documents_textes.pdf', 'page': 4, 'creationdate': '2023-10-30T09:46:28+01:00', 'total_pages': 35, 'creator': 'Acrobat PDFMaker 22 pour PowerPoint', 'producer': 'Adobe PDF Library 22.3.39'}, page_content='II. Techniques de prétraitement\n1.Lowercasing/uppercasing\n(Minuscule/Majuscule)\n2.Noise removal (Suppression du bruit)\n3.Text normalization (Normalisation du texte)\n4.Stemming (Dérivé)\n5.Lemmatization (Lemmatisation)\n6.Tokenization(Tokenisation)\n7.Stop words (Mots vides)\nProf: R. EL AYACHI\n5\nComputer Vision'), Document(metadata={'source': './pdfs/Chapitre_3_Prétraitement_des_documents_textes.pdf', 'page': 3, 'author': 'INFORMATIQUE', 'page_label': '4', 'total_pages': 35, 'creationdate': '2023-10-30T09:46:28+01:00', 'creato

Maintenant, pour la partie RAG, on va créer notre prompt système (*system prompt*).


# 4.  *Prompt Design*

## 4.1 Prompt template

In [48]:
prompt_template="""
Answer the following question based only on provided context The context is about Prétraitement des documents textes
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

### Note sur le contexte
Vous pouvez ajouter cette précision si vous savez que votre document concerne la Direction des études et des prévisions financières (DEPF). Vous indiquez alors le contexte suivant : *« Le contexte concerne la Note de conjoncture de la Direction des études et des prévisions financières de décembre 2025. »* C'est une phrase utile. Si vous préférez que le prompt reste général, quel que soit le fichier PDF, vous pouvez simplement retirer cette partie.

Le contexte est délimité par la balise `<context>`, et la question par la balise `<question>`.

### Gestion des hallucinations
Pour éviter les hallucinations, spécifiez dans le prompt : *« Si la réponse ne figure pas dans ce contexte, répondez par : "Je ne sais pas !" »*. Cela permet de régler efficacement le problème des hallucinations.


## 4.2 *Retrieving the Relevant Documents*

### Passage à la partie *retrieving*

Voici ma démarche : je récupère les documents, puis je les **convertis**. Pour chaque document (`relevant_document_chunks`), je sélectionne uniquement le contenu de la page (`d.page_content`), ce qui me génère une liste de textes. Enfin, nous allons concaténer cette liste de *chunks* de texte en utilisant le séparateur `". "`.


In [49]:
user_input="C'est quo la mesure de similarité ? "

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query

'1°/ Définition:\nMesurer la similarité entre deux phrases\nconsiste à évaluer jusqu’à quel point le sens de\nces phrases est proche en termes de proximité\nde surface “similarité syntaxique” et de\nsignification “similarité sémantique”.\nProf: R. EL AYACHI\n31\nIV. Mesures de similarité\nComputer Vision. 3°/ Similarités sémantiques :\n• Similarité de Wu et Palmer:\nProf: R. EL AYACHI\n33\nIV. Mesures de similarité\nComputer Vision\n𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆 𝑥𝑥, 𝑦𝑦 = 2 × 𝑁𝑁3\n𝑁𝑁1 + 𝑁𝑁2\nR\nCS\nx y\nN1 N2\nN3. 1. Définition\n2. Exemples des mesures\nProf: R. EL AYACHI\n30\nIV. Mesures de similarité\nComputer Vision. 2°/ Similarités syntaxiques :\n• Distance Euclidienne:\n• Distance Cosinus:\n• Distance Bray-Curtis:\nProf: R. EL AYACHI\n32\nIV. Mesures de similarité\nComputer Vision\n𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆 𝑥𝑥, 𝑦𝑦 = �\n𝑖𝑖=1\n𝑛𝑛\n𝑥𝑥𝑖𝑖 − 𝑦𝑦𝑖𝑖 2\n𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆 𝑥𝑥, 𝑦𝑦 = ∑𝑖𝑖=1𝑛𝑛 𝑥𝑥𝑖𝑖 𝑦𝑦𝑖𝑖'

In [50]:
len(relevant_document_chunks)

4

## 4.3 Le prompt

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .




In [51]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on provided context The context is about Prétraitement des documents textes
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
1°/ Définition:
Mesurer la similarité entre deux phrases
consiste à évaluer jusqu’à quel point le sens de
ces phrases est proche en termes de proximité
de surface “similarité syntaxique” et de
signification “similarité sémantique”.
Prof: R. EL AYACHI
31
IV. Mesures de similarité
Computer Vision. 3°/ Similarités sémantiques :
• Similarité de Wu et Palmer:
Prof: R. EL AYACHI
33
IV. Mesures de similarité
Computer Vision
𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆𝑆 𝑥𝑥, 𝑦𝑦 = 2 × 𝑁𝑁3
𝑁𝑁1 + 𝑁𝑁2
R
CS
x y
N1 N2
N3. 1. Définition
2. Exemples des mesures
Prof: R. EL AYACHI
30
IV. Mesures de similarité
Computer Vision. 2°/ Similarités syntaxiques :
• Distance Euclidienne:
• Distance Cosinus:
• Distance Bray-Curtis:
Prof: R. EL AYACHI
32
I

Donc, ça c'est le prompt.

# 5. *Récupérer la réponse de la part de llm*

 Maintenant on va utiliser le prompt. On va poser cette question à LLM.

In [52]:

resp=llm.invoke(prompt)
from IPython.display import Markdown
print(resp.content)

Mesurer la similarité entre deux phrases consiste à évaluer jusqu’à quel point le sens de ces phrases est proche en termes de proximité de surface “similarité syntaxique” et de signification “similarité sémantique”.


## 6. Regroupement dans une application RAG


### Définition de la fonction RAG

Nous allons définir `RAG` comme notre fonction principale de RAG. Elle prendra en paramètres : `query`, `llm` et `prompt_template`.

**Comment fonctionne cette fonction RAG ?**
1. **Le Retriever** : On lui passe la question (`query`), il recherche les documents pertinents et les convertit en une liste de textes. Chaque texte représente le contenu d'une page (`page_content`).
2. **La Concaténation** : Nous allons concaténer cette liste de textes à l'aide d'un séparateur. Ce texte final devient notre **contexte**.
3. **La Génération** : On injecte ce contexte et la question dans le `prompt_template`. On appelle ensuite la méthode `invoke()` et on retourne le contenu généré.

Voici donc la fonction qui nous intéresse. Pour pouvoir la tester, il vous suffira de faire appel à `RAG()` à chaque fois que vous voudrez lui poser une question.



In [18]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG 

In [53]:
resp=RAG(" c'est quoi le JWT pour un projet spring security ")
print(resp)

Je ne sais pas!


# 7. Évaluation de la réponse du LLM


## 7.1 Architecture

![Logo](./rag_juge.gif)

In [54]:
#question ou query 
user_input=" Qui sont les techniques de prétraitement "

Récupérer le context 

In [55]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [56]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [57]:
# Réponse d'un RAG
answer=RAG(user_input)
print(display(Markdown(answer)))

Les techniques de prétraitement sont :
1. Lowercasing/uppercasing (Minuscule/Majuscule)
2. Noise removal (Suppression du bruit)
3. Text normalization (Normalisation du texte)
4. Stemming (Dérivé)
5. Lemmatization (Lemmatisation)
6. Tokenization (Tokenisation)
7. Stop words (Mots vides)

None


## 7.2- *Groundness*

Le prompt pour le juge 

In [24]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [58]:
groundness_checker = ChatOllama(
    model="gemma4:31b-cloud",   # ou glm-5.2, kimi-k2.6, gpt-oss:120b
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": f"Bearer {api_key}"}},
    temperature=0,
)

In [59]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content



In [60]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

**Étapes nécessaires pour évaluer la réponse :**

1. **Analyse de la question :** Identifier l'information spécifique demandée par l'utilisateur (ici, la liste des techniques de prétraitement).
2. **Analyse du contexte :** Repérer dans le texte fourni les segments qui traitent des "techniques de prétraitement".
3. **Comparaison :** Vérifier si chaque élément mentionné dans la réponse figure explicitement dans le contexte.
4. **Vérification d'extrapolations :** S'assurer que l'IA n'a pas ajouté d'informations extérieures (connaissances générales) non présentes dans le texte source.
5. **Attribution du score :** Noter la réponse selon l'échelle de 1 à 5 en fonction de la fidélité exclusive au contexte.

---

**Explication étape par étape :**

*   **Étape 1 :** La question demande "Qui sont les techniques de prétraitement" (malgré la maladresse syntaxique, on comprend qu'il s'agit de demander "Quelles sont...").
*   **Étape 2 :** Dans le contexte, sous la section "II. Techniques de prétraitement", on trouve une liste numérotée de 1 à 7 :
    1. Lowercasing/uppercasing (Minuscule/Majuscule)
    2. Noise removal (Suppression du bruit)
    3. Text normalization (Normalisation du texte)
    4. Stemming (Dérivé)
    5. Lemmatization (Lemmatisation)
    6. Tokenization (Tokenisation)
    7. Stop words (Mots vides)
*   **Étape 3 :** La réponse de l'IA liste exactement ces 7 points, dans le même ordre et avec les mêmes termes (anglais et français).
*   **Étape 4 :** Aucun élément supplémentaire n'a été ajouté. L'IA ne s'est pas servie de connaissances externes pour compléter la liste ou définir les termes.

---

**Évaluation :**
La réponse est une extraction directe et fidèle des informations contenues dans le contexte. Elle ne contient aucune information étrangère au texte fourni.

**Score : 5**
**La métrique est entièrement respectée.**

None


### Échelle de notation pour la fidélité au contexte (Faithfulness)


| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |
